In [0]:

from pyspark.sql import functions as F
from functools import reduce
import uuid

CATALOG = "workspace"
BRONZE_SCHEMA = "codi_bronze"
SILVER_SCHEMA = "codi_silver"

YEARS = [2018, 2019, 2020]

TARGET_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}."
    "deis_egresos_historico"
)

EXPECTED_COUNTS = {
    2018: 1669602,
    2019: 1667180,
    2020: 1330477
}

EXPECTED_TOTAL = sum(EXPECTED_COUNTS.values())

METADATA_COLUMNS = {
    "_source_system",
    "_source_year",
    "_source_file",
    "_source_sha256",
    "_batch_id",
    "_ingestion_timestamp"
}

print("Destino:", TARGET_TABLE)
print("Años:", YEARS)
print("Registros esperados:", f"{EXPECTED_TOTAL:,}")


In [0]:

datasets = []
columnas_base = None

for year in YEARS:

    source_table = (
        f"{CATALOG}.{BRONZE_SCHEMA}."
        f"deis_egresos_{year}"
    )

    if not spark.catalog.tableExists(source_table):
        raise RuntimeError(
            f"Tabla Bronze no encontrada: {source_table}"
        )

    df = spark.table(source_table)

    # Homologación del nombre distinto en 2019
    if "PERTENENCIA_ESTABLECIMIENTO_SALU" in df.columns:

        if "PERTENENCIA_ESTABLECIMIENTO_SALUD" in df.columns:
            raise ValueError("Columnas duplicadas al homologar")

        df = df.withColumnRenamed(
            "PERTENENCIA_ESTABLECIMIENTO_SALU",
            "PERTENENCIA_ESTABLECIMIENTO_SALUD"
        )

    columnas_negocio = {
        c for c in df.columns
        if c not in METADATA_COLUMNS
    }

    if columnas_base is None:
        columnas_base = columnas_negocio

    elif columnas_negocio != columnas_base:
        raise ValueError(
            f"Esquema incompatible en {year}. "
            f"Faltantes: {columnas_base-columnas_negocio}. "
            f"Adicionales: {columnas_negocio-columnas_base}"
        )

    # Asegurar la procedencia del archivo
    df = df.withColumn(
        "_source_year", F.lit(year)
    )

    datasets.append(df)

    print(
        f"{year}: {len(columnas_negocio)} "
        "columnas de negocio"
    )

# Unir los 3 años sin descartar registros
df_silver = reduce(
    lambda a, b: a.unionByName(
        b, allowMissingColumns=True
    ),
    datasets
)

print("Consolidación lógica preparada")


In [0]:

anio_original = F.trim(
    F.col("ANO_EGRESO").cast("string")
)

df_silver = (
    df_silver

    # Año numérico solo si tiene formato YYYY
    .withColumn(
        "anio_egreso_normalizado",
        F.when(
            anio_original.rlike(r"^[0-9]{4}$"),
            anio_original.cast("int")
        ).otherwise(F.lit(None).cast("int"))
    )

    # Clasificación sin interpretar el significado de *
    .withColumn(
        "estado_calidad_anio",
        F.when(
            anio_original == "*",
            F.lit("VALOR_ESPECIAL_NO_DOCUMENTADO")
        )
        .when(
            anio_original.isNull() |
            (anio_original == ""),
            F.lit("NULO_O_VACIO")
        )
        .when(
            F.col("anio_egreso_normalizado").isNull(),
            F.lit("FORMATO_NO_RECONOCIDO")
        )
        .when(
            F.col("anio_egreso_normalizado")
            != F.col("_source_year"),
            F.lit("DISTINTO_ANIO_ARCHIVO")
        )
        .otherwise(F.lit("VALIDO"))
    )

    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

print("Campos derivados incorporados")

display(
    df_silver
    .groupBy(
        "_source_year",
        "estado_calidad_anio"
    )
    .count()
    .orderBy("_source_year", "estado_calidad_anio")
)


In [0]:

# Contabilizar Silver por año de procedencia
conteos_silver = {
    int(row["_source_year"]): row["count"]
    for row in (
        df_silver
        .groupBy("_source_year")
        .count()
        .collect()
    )
}

resultados = []

for year in YEARS:

    source_table = (
        f"{CATALOG}.{BRONZE_SCHEMA}."
        f"deis_egresos_{year}"
    )

    bronze_count = spark.table(source_table).count()
    silver_count = conteos_silver.get(year, 0)

    diferencia = bronze_count - silver_count

    resultados.append({
        "anio": year,
        "bronze": bronze_count,
        "silver": silver_count,
        "diferencia": diferencia,
        "estado": "OK" if diferencia == 0 else "ERROR"
    })

df_conciliacion = spark.createDataFrame(resultados)

display(df_conciliacion.orderBy("anio"))

# Verificar todos los años
assert all(
    r["diferencia"] == 0 for r in resultados
), "ERROR: descuadre entre Bronze y Silver"

total_silver = sum(conteos_silver.values())

assert total_silver == EXPECTED_TOTAL, (
    f"Total inesperado: {total_silver:,}"
)

print("CONCILIACIÓN EXITOSA")
print(f"Total Silver: {total_silver:,}")


In [0]:

df_calidad = (
    df_silver
    .groupBy(
        "_source_year",
        "estado_calidad_anio"
    )
    .agg(F.count("*").alias("registros"))
)

display(
    df_calidad.orderBy(
        "_source_year",
        "estado_calidad_anio"
    )
)

# Valores especiales observados previamente
EXPECTED_SPECIAL = {
    2018: 49152,
    2019: 43845,
    2020: 37542
}

especiales = {
    int(r["_source_year"]): r["registros"]
    for r in df_calidad.filter(
        F.col("estado_calidad_anio")
        == "VALOR_ESPECIAL_NO_DOCUMENTADO"
    ).collect()
}

for year in YEARS:
    actual = especiales.get(year, 0)
    expected = EXPECTED_SPECIAL[year]

    assert actual == expected, (
        f"{year}: esperados {expected}, "
        f"encontrados {actual}"
    )

print("Control de valores especiales: OK")
print("No se eliminaron registros con '*'")


In [0]:

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{CATALOG}.{SILVER_SCHEMA}"
)

if spark.catalog.tableExists(TARGET_TABLE):

    print("La tabla Silver ya existe.")
    print("No se modificaron sus datos.")

else:

    (
        df_silver.write
        .format("delta")
        .mode("append")
        .saveAsTable(TARGET_TABLE)
    )

    print("Tabla Silver creada:", TARGET_TABLE)

# Validar tabla persistida
df_persistido = spark.table(TARGET_TABLE)

total_persistido = df_persistido.count()

assert total_persistido == EXPECTED_TOTAL, (
    f"Total persistido inesperado: "
    f"{total_persistido:,}"
)

display(
    df_persistido
    .groupBy("_source_year")
    .agg(F.count("*").alias("registros"))
    .orderBy("_source_year")
)

print("Total persistido:", f"{total_persistido:,}")
print("SILVER COMPLETADO")


In [0]:

display(
    spark.sql(f"DESCRIBE DETAIL {TARGET_TABLE}")
)

display(
    spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE}")
)
